# 01 - WLT-OSN Dataset Validation
**Project:** Illegal Wildlife Trade Detection System  
**Dataset:** Mou et al. (ICWSM 2024) `share_data.csv` (Ivory-focused online trade listings)  

### Goals:
1. Load and clean the raw dataset using `load_wlt_osn`.
2. Inspect and document extreme class imbalance (~3% positives).
3. Check for exact and near-duplicates using TF-IDF cosine similarity.
4. Analyze source group distribution and visualize label concentration.
5. Verify group-aware split to ensure zero label leakage between train and test sets.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure backend is in python path
REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT / "backend"))

from data.loader import load_wlt_osn, group_aware_split
from config import settings

## 1. Load and Clean Dataset
We load `data/raw/share_data.csv` and deduplicate exact text hashes.

In [ ]:
csv_path = REPO_ROOT / "data" / "raw" / "share_data.csv"
df = load_wlt_osn(csv_path)
print(f"Total Cleaned Rows: {len(df)}")
df.head()

## 2. Class Imbalance Analysis

In [ ]:
class_counts = df['label'].value_counts()
positives = class_counts.get(1, 0)
negatives = class_counts.get(0, 0)
pos_pct = (positives / len(df)) * 100

print(f"Negatives (0): {negatives} ({100 - pos_pct:.2f}%)")
print(f"Positives (1): {positives} ({pos_pct:.2f}%)")
print(f"Imbalance Ratio: ~1:{negatives // positives}")

## 3. Near-Duplicate Analysis (Cosine Similarity > 0.95)

In [ ]:
# Vectorize with TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_mat = vectorizer.fit_transform(df['text'])

# Sample positive subset for pair-wise comparison
pos_idx = df[df['label'] == 1].index
pos_tfidf = tfidf_mat[pos_idx]
sim_mat = cosine_similarity(pos_tfidf, pos_tfidf)
np.fill_diagonal(sim_mat, 0)

high_sim_pairs = np.argwhere(sim_mat > 0.95)
print(f"Near-duplicate pairs found in positives (sim > 0.95): {len(high_sim_pairs) // 2}")

## 4. Source Group Distribution
Analyze and plot the top 15 sources to observe positive label concentration.

In [ ]:
top_sources = df['source_group'].value_counts().head(15)
source_pos = df.groupby('source_group')['label'].sum().loc[top_sources.index]

fig, ax = plt.subplots(figsize=(12, 6))
top_sources.plot(kind='bar', ax=ax, color='steelblue', label='Total Listings')
source_pos.plot(kind='bar', ax=ax, color='crimson', label='Suspected Trade (Positives)')
ax.set_title("Distribution of Top 15 Source Groups in WLT-OSN Dataset")
ax.set_xlabel("Source Group")
ax.set_ylabel("Number of Records")
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Group-Aware Split Verification
Crucial integrity gate: Ensure zero overlap between source groups in training and test sets.

In [ ]:
train_df, test_df = group_aware_split(df, test_size=0.2, seed=42)

train_groups = set(train_df['source_group'].unique())
test_groups = set(test_df['source_group'].unique())
leakage = train_groups.intersection(test_groups)

print(f"Train size: {len(train_df)} ({train_df['label'].sum()} positives)")
print(f"Test size: {len(test_df)} ({test_df['label'].sum()} positives)")
print(f"Source group leakage count: {len(leakage)}")
assert len(leakage) == 0, "CRITICAL FAILURE: Source group leakage detected!"
print("PASSED: Zero source group leakage verified.")